In [1]:
import pandas as pd
import altair as alt

df = pd.read_csv('videogamesales.csv')
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)]

genre_region = (
    df.groupby('Genre')[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']]
    .sum()
    .reset_index())

genre_region['Total'] = genre_region[['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales']].sum(axis=1)

for region, col in [('NA', 'NA_Sales'), ('EU', 'EU_Sales'), ('JP', 'JP_Sales'), ('Other', 'Other_Sales')]:
    genre_region[f'{region}_Pct'] = (genre_region[col] / genre_region['Total'] * 100).round(1)

genre_long = genre_region.melt(
    id_vars=['Genre', 'Total', 'NA_Pct', 'EU_Pct', 'JP_Pct', 'Other_Pct'],
    value_vars=['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales'],
    var_name='Region',
    value_name='Sales')

genre_long['Region'] = genre_long['Region'].map({
    'NA_Sales': 'North America',
    'EU_Sales': 'Europe',
    'JP_Sales': 'Japan',
    'Other_Sales': 'Other'})

genre_long['Pct'] = genre_long.apply(
    lambda row: genre_region.loc[
        genre_region['Genre'] == row['Genre'],
        {'North America': 'NA_Pct', 'Europe': 'EU_Pct', 'Japan': 'JP_Pct', 'Other': 'Other_Pct'}[row['Region']]
    ].values[0], axis=1)

sort_options = {
    key: genre_region.sort_values(col, ascending=False)['Genre'].tolist()
    for key, col in [('Total Sales', 'Total'), ('NA Sales', 'NA_Sales'), ('EU Sales', 'EU_Sales'), ('JP Sales', 'JP_Sales')]
}

color_map = {
    'North America': '#4C78A8',
    'Europe':        '#F58518',
    'Japan':         '#E45756',
    'Other':         '#72B7B2'}

highlight = alt.selection_point(fields=['Genre'], on='click', empty='all')

bars = (
    alt.Chart(genre_long)
    .mark_bar()
    .encode(
        x=alt.X('Sales:Q', title='Total Sales (millions)'),
        y=alt.Y('Genre:N', title='Genre', sort=sort_options['Total Sales']),
        color=alt.Color('Region:N',
            scale=alt.Scale(domain=list(color_map.keys()), range=list(color_map.values())),
            legend=alt.Legend(title='Region')),
        opacity=alt.condition(highlight, alt.value(1.0), alt.value(0.3)),
        tooltip=[
            alt.Tooltip('Genre:N', title='Genre'),
            alt.Tooltip('Region:N', title='Region'),
            alt.Tooltip('Sales:Q', title='Sales (M)', format='.1f'),
            alt.Tooltip('Pct:Q', title='% of Genre Total', format='.1f'),
            alt.Tooltip('Total:Q', title='Genre Total (M)', format='.1f')])
    .add_params(highlight)
    .properties(width=700, height=400, title='Video Game Sales by Genre and Region'))

note = (
    alt.Chart({'values': [{}]})
    .mark_text(align='right', color='gray', fontSize=11)
    .encode(
        x=alt.value(700),
        y=alt.value(-10),
        text=alt.value('Click to highlight · Double-click to reset')))

(bars + note)


alt.LayerChart(...)